# 03 · Overfitting y validación

## Objetivo

En este notebook vamos a estudiar uno de los conceptos más importantes de Machine Learning:

> Un modelo puede funcionar extremadamente bien con los datos de entrenamiento y, sin embargo, funcionar mal con datos nuevos.

Eso es **overfitting** o **sobreajuste**.

Usaremos un problema sintético inspirado en ATM: predecir si una ventana operativa de 15 minutos tendrá **alta carga de coordinación** a partir de varias variables.

La idea principal será distinguir claramente tres conjuntos:

- **train**: para ajustar los parámetros del modelo;
- **validation**: para comparar modelos e hiperparámetros;
- **test**: para estimar el rendimiento final una vez terminadas las decisiones.

Al terminar deberías entender:

- qué es overfitting;
- qué es underfitting;
- por qué no basta con medir el rendimiento en train;
- por qué necesitamos validación;
- por qué el conjunto de test no debe usarse continuamente;
- cómo influye la complejidad de un modelo;
- cómo detectar una brecha entre train y validation;
- y cómo elegir un hiperparámetro sin contaminar el test.


## 1. Importaciones


In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 2. Crear datos ATM sintéticos

Vamos a generar varias features:

- `flights_15min`: vuelos en una ventana de 15 min;
- `conflicts_predicted`: conflictos potenciales previstos;
- `vertical_movements`: ascensos/descensos;
- `weather_severity`: severidad meteorológica sintética;
- `sector_transitions`: entradas/salidas entre sectores;
- `noise_feature`: variable aleatoria sin valor predictivo.

El target será:

- `0`: carga normal;
- `1`: alta carga.

La relación real será relativamente sencilla, pero añadiremos ruido para que no sea perfectamente separable.


In [ ]:
rng = np.random.default_rng(42)

n_samples = 1200

flights_15min = rng.integers(5, 36, size=n_samples)
conflicts_predicted = rng.poisson(lam=2.2, size=n_samples)
vertical_movements = rng.integers(0, 12, size=n_samples)
weather_severity = rng.uniform(0, 1, size=n_samples)
sector_transitions = rng.integers(0, 10, size=n_samples)

# Esta variable es deliberadamente ruido puro.
noise_feature = rng.normal(0, 1, size=n_samples)

# Regla latente sintética.
score = (
    -7.5
    + 0.22 * flights_15min
    + 0.65 * conflicts_predicted
    + 0.10 * vertical_movements
    + 1.8 * weather_severity
    + 0.08 * sector_transitions
)

prob_high_load = 1 / (1 + np.exp(-score))
high_load = rng.binomial(1, prob_high_load)

df = pd.DataFrame({
    "flights_15min": flights_15min,
    "conflicts_predicted": conflicts_predicted,
    "vertical_movements": vertical_movements,
    "weather_severity": weather_severity,
    "sector_transitions": sector_transitions,
    "noise_feature": noise_feature,
    "high_load": high_load
})

df.head()


## 3. Exploración inicial


In [ ]:
df.describe()


In [ ]:
df["high_load"].value_counts(normalize=True).sort_index()


No buscamos que el dataset sea perfecto. De hecho, si fuera perfectamente separable sería difícil mostrar correctamente el problema de generalización.


## 4. Separar features y target


In [ ]:
features = [
    "flights_15min",
    "conflicts_predicted",
    "vertical_movements",
    "weather_severity",
    "sector_transitions",
    "noise_feature",
]

X = df[features]
y = df["high_load"]

print("X:", X.shape)
print("y:", y.shape)


# 5. Train / validation / test

Primero reservaremos un **20 % para test**.

Después dividiremos el 80 % restante en:

- 60 % total para train;
- 20 % total para validation;
- 20 % total para test.

El test debe quedar apartado hasta el final.


In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,   # 25 % de 80 % = 20 % total
    random_state=42,
    stratify=y_train_val
)

print("Train     :", X_train.shape[0])
print("Validation:", X_val.shape[0])
print("Test      :", X_test.shape[0])


In [ ]:
split_summary = pd.DataFrame({
    "subset": ["train", "validation", "test"],
    "samples": [len(y_train), len(y_val), len(y_test)],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()]
})

split_summary


## 6. Un modelo deliberadamente simple

Empezamos con un árbol muy poco profundo.

Un árbol con poca profundidad tiene poca capacidad para aprender reglas complejas.


In [ ]:
simple_tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

simple_tree.fit(X_train, y_train)

train_pred_simple = simple_tree.predict(X_train)
val_pred_simple = simple_tree.predict(X_val)

print("Accuracy train     :", round(accuracy_score(y_train, train_pred_simple), 3))
print("Accuracy validation:", round(accuracy_score(y_val, val_pred_simple), 3))


Si ambos resultados son relativamente modestos y parecidos, podemos estar ante cierto **underfitting**:

> el modelo no tiene suficiente capacidad para capturar bien la relación existente en los datos.


## 7. Un modelo extremadamente flexible

Ahora permitimos que el árbol crezca prácticamente sin restricciones.


In [ ]:
deep_tree = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

deep_tree.fit(X_train, y_train)

train_pred_deep = deep_tree.predict(X_train)
val_pred_deep = deep_tree.predict(X_val)

print("Accuracy train     :", round(accuracy_score(y_train, train_pred_deep), 3))
print("Accuracy validation:", round(accuracy_score(y_val, val_pred_deep), 3))


Es habitual que el árbol sin restricciones alcance una accuracy de entrenamiento extremadamente alta, incluso cercana a 1.0.

Pero eso **no significa que sea mejor**.

La pregunta importante es:

> ¿qué ocurre con datos que el modelo no ha visto?


# 8. Comparar complejidad del modelo

Vamos a entrenar árboles con distintas profundidades y medir:

- accuracy en train;
- accuracy en validation.

`max_depth` será nuestro hiperparámetro de complejidad.


In [ ]:
depths = list(range(1, 21))

rows = []

for depth in depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    rows.append({
        "max_depth": depth,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "val_accuracy": accuracy_score(y_val, val_pred),
    })

depth_results = pd.DataFrame(rows)
depth_results


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    depth_results["max_depth"],
    depth_results["train_accuracy"],
    marker="o",
    label="Train"
)
plt.plot(
    depth_results["max_depth"],
    depth_results["val_accuracy"],
    marker="o",
    label="Validation"
)

plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Complejidad del modelo: train vs validation")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


Este gráfico resume el problema:

- con muy poca profundidad, train y validation pueden rendir mal → **underfitting**;
- al aumentar la capacidad, ambos mejoran;
- llega un punto en que train sigue mejorando, pero validation deja de hacerlo o empeora → **overfitting**.

La diferencia entre train y validation se suele llamar **generalization gap**.


## 9. Calcular el generalization gap


In [ ]:
depth_results["generalization_gap"] = (
    depth_results["train_accuracy"] - depth_results["val_accuracy"]
)

depth_results


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    depth_results["max_depth"],
    depth_results["generalization_gap"],
    marker="o"
)
plt.axhline(0, linestyle="--")
plt.xlabel("max_depth")
plt.ylabel("Train accuracy - Validation accuracy")
plt.title("Generalization gap")
plt.grid(alpha=0.2)
plt.show()


Un gap grande no implica por sí solo que el modelo sea inútil, pero es una señal clara de que debemos vigilar la generalización.


# 10. Elegir el hiperparámetro usando validation

Seleccionamos la profundidad con mejor accuracy de validation.

Todavía **no usamos el test**.


In [ ]:
best_row = depth_results.loc[depth_results["val_accuracy"].idxmax()]
best_depth = int(best_row["max_depth"])

print("Mejor max_depth según validation:", best_depth)
print("Validation accuracy:", round(best_row["val_accuracy"], 3))
print("Train accuracy     :", round(best_row["train_accuracy"], 3))


Este es el flujo correcto:

```text
train      -> ajustar parámetros
validation -> elegir hiperparámetros/modelo
test       -> evaluación final
```

Si miramos continuamente el test para decidir qué modelo usar, el test deja de ser realmente independiente.


## 11. Entrenar el modelo elegido

Ahora que ya hemos decidido `max_depth`, podemos volver a entrenar usando juntos train + validation.

Eso permite aprovechar más datos antes de la evaluación final.


In [ ]:
final_model = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
)

final_model.fit(X_train_val, y_train_val)

test_pred = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]


## 12. Evaluación final sobre test


In [ ]:
test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "f1": f1_score(y_test, test_pred),
    "roc_auc": roc_auc_score(y_test, test_proba),
}

for name, value in test_metrics.items():
    print(f"{name:10s}: {value:.3f}")


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    display_labels=["Normal", "Alta carga"]
)

plt.title("Modelo final sobre test")
plt.show()


Esta es la primera vez que el conjunto de test interviene en nuestra decisión experimental.

Idealmente, una vez visto el resultado de test, no deberíamos seguir ajustando repetidamente el modelo basándonos en él.


# 13. Comparar modelo elegido y árbol sobreajustado

Vamos a comparar ambos directamente sobre test.


In [ ]:
overfit_model = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

overfit_model.fit(X_train_val, y_train_val)

overfit_test_pred = overfit_model.predict(X_test)
overfit_test_proba = overfit_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "model": [
        f"Árbol seleccionado (depth={best_depth})",
        "Árbol sin límite"
    ],
    "train_val_accuracy": [
        accuracy_score(y_train_val, final_model.predict(X_train_val)),
        accuracy_score(y_train_val, overfit_model.predict(X_train_val)),
    ],
    "test_accuracy": [
        accuracy_score(y_test, test_pred),
        accuracy_score(y_test, overfit_test_pred),
    ],
    "test_f1": [
        f1_score(y_test, test_pred),
        f1_score(y_test, overfit_test_pred),
    ],
    "test_roc_auc": [
        roc_auc_score(y_test, test_proba),
        roc_auc_score(y_test, overfit_test_proba),
    ]
})

comparison


Fíjate especialmente en esta situación típica:

```text
modelo A:
train = excelente
test  = peor

modelo B:
train = algo peor
test  = mejor
```

El modelo B suele ser preferible.

Machine Learning no consiste en memorizar el conjunto de entrenamiento, sino en **generalizar**.


# 14. ¿Puede el modelo aprender ruido?

Incluimos deliberadamente esta feature:

```text
noise_feature
```

Es una variable aleatoria que no participa en la generación del target.

Un modelo flexible puede encontrar patrones accidentales en ella.


In [ ]:
feature_importance = pd.DataFrame({
    "feature": features,
    "selected_tree_importance": final_model.feature_importances_,
    "overfit_tree_importance": overfit_model.feature_importances_,
}).sort_values("overfit_tree_importance", ascending=False)

feature_importance


Que un árbol muy complejo asigne importancia a una variable de ruido es una buena intuición de lo que significa sobreajustar:

> aprender regularidades accidentales del dataset concreto en vez de la estructura general del problema.


# 15. Curva de aprendizaje

Otra herramienta útil consiste en observar cómo cambia el rendimiento cuando aumentamos el número de muestras de entrenamiento.

Usaremos el modelo seleccionado y validación cruzada interna.


In [ ]:
learning_model = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
)

train_sizes, train_scores, val_scores = learning_curve(
    learning_model,
    X_train_val,
    y_train_val,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

learning_df = pd.DataFrame({
    "train_samples": train_sizes,
    "train_accuracy": train_mean,
    "validation_accuracy": val_mean,
})

learning_df


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    learning_df["train_samples"],
    learning_df["train_accuracy"],
    marker="o",
    label="Train"
)

plt.plot(
    learning_df["train_samples"],
    learning_df["validation_accuracy"],
    marker="o",
    label="Validation"
)

plt.xlabel("Número de muestras de entrenamiento")
plt.ylabel("Accuracy")
plt.title("Curva de aprendizaje")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


Interpretación aproximada:

### Train alto, validation claramente inferior
Puede indicar **alta varianza / overfitting**.

Más datos pueden ayudar.

### Train y validation ambos bajos
Puede indicar **alto sesgo / underfitting**.

Más datos quizá no solucionen el problema: podría hacer falta un modelo más adecuado o mejores features.

### Ambos convergen en un valor razonablemente alto
Es una señal favorable de generalización.


# 16. Validation frente a cross-validation

En este notebook usamos un único conjunto de validation porque permite ver claramente el concepto.

En proyectos reales, especialmente con pocos datos, es habitual usar **cross-validation**.

Por ejemplo:

```text
Fold 1: validation | train train train train
Fold 2: train | validation | train train train
Fold 3: train train | validation | train train
...
```

La ventaja es que la estimación depende menos de una única partición aleatoria.

Pero el concepto sigue siendo el mismo:

> los datos utilizados para evaluar una decisión no deben ser exactamente los mismos con los que ajustamos el modelo.


# 17. Data leakage

Existe un problema relacionado pero diferente: **data leakage**.

Ocurre cuando información que el modelo no debería conocer durante la predicción entra accidentalmente en el entrenamiento.

Ejemplos:

- usar una variable calculada después de producirse el evento;
- normalizar todo el dataset antes de separar train/test;
- incluir en train observaciones casi idénticas a las de test;
- usar información futura para predecir el pasado.

Un modelo con leakage puede obtener métricas espectaculares y ser completamente inválido en producción.

Esto será especialmente importante cuando trabajemos con datos temporales ATM.


# 18. Ejercicios

### Ejercicio 1

Cambia el rango de profundidades:

```python
range(1, 40)
```

¿A partir de qué profundidad ves claramente que train y validation divergen?

### Ejercicio 2

Elimina `noise_feature` y repite el experimento.

¿Cambia el comportamiento del árbol profundo?

### Ejercicio 3

Reduce el dataset:

```python
n_samples = 200
```

¿Se vuelve más fácil sobreajustar?

¿Por qué?

### Ejercicio 4

Aumenta el dataset:

```python
n_samples = 5000
```

¿Cambia la brecha train-validation?

### Ejercicio 5

En vez de seleccionar el modelo por `accuracy`, selecciónalo por `f1`.

¿Obtienes la misma profundidad?

### Ejercicio 6

Imagina que has probado 50 modelos diferentes y has elegido el que mejor funciona en el test.

¿Sigue siendo el test una estimación independiente del rendimiento?

Explica por qué.


# 19. Ideas clave

Al terminar este notebook deberías poder explicar:

1. Qué es **underfitting**.
2. Qué es **overfitting**.
3. Por qué una métrica de entrenamiento excelente no garantiza un buen modelo.
4. Qué diferencia existe entre train, validation y test.
5. Por qué los hiperparámetros deben elegirse usando validation, no test.
6. Qué significa **generalization gap**.
7. Cómo influye la complejidad del modelo.
8. Por qué un modelo puede aprender ruido.
9. Para qué sirve una curva de aprendizaje.
10. Qué es data leakage y por qué puede invalidar un experimento.

## La idea central

```text
Entrenar bien ≠ generalizar bien
```

Nuestro objetivo real es:

```text
buen rendimiento sobre datos nuevos
```

## Siguiente notebook

`04_neural_network.ipynb`

Usaremos un problema no lineal para entender por qué una red neuronal puede representar relaciones que un modelo lineal simple no puede capturar.
